## SRP423519

**paper:** [PMID: 38388772](https://pmc.ncbi.nlm.nih.gov/articles/PMC10883927/) - Establishment and characterization of turtle liver organoids provides a potential model to decode their unique adaptations, 2024

**date, curator:** 2026-04-30, Sara Carsanaro

**notes**
* rejected organoid samples, including single-cell sample. only annotated liver tissue samples
* libraries did not come from SRA correctly, seems to be an issue with how they were submitted, there may be problems with the pipeline
* supplementary table 1 has sample data including age, sex

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,liver,UBERON:0002107,liver,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,juvenile - 1 year,UBERON:0034919,juvenile stage,perfect match
1,adult,UBERON:0000113,post-juvenile adult stage,perfect match
3,hatchling,UBERON:0007221,neonate stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP423519"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [4]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 15/15 [00:15<00:00,  1.03s/it]
14 samples dont have attributes, try to find them somewhere else
  0%|                                                    | 0/14 [00:00<?, ?it/s] WARNING:  FAILURE ( Fri May  1 11:21:15 CEST 2026 )
nquire -url https://eutils.ncbi.nlm.nih.gov/entrez/eutils/ esummary.fcgi -db biosample -id SAMN29907574 -version 2.0 -retmode xml -tool edirect -edirect 16.2 -edirect_os Darwin -email scarsana@SORGE42778
<ERROR>Invalid uid SAMN29907574 at position= 0</ERROR>
SECOND ATTEMPT
nquire -url https://eutils.ncbi.nlm.nih.gov/entrez/eu

### library annnotations

In [5]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19452171,SRP423519,Illumina HiSeq 4000,SRS14105880,,,UBERON:0034919,juvenile stage,,liver,juvenile - 1 year,,,perfect match,M,,,8475,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,Turtle_4_Liver_Tissue,SAMN29907573,1,year,,,,,,,,,01/05/2026,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limited cycle PCR being used for library enrichment. The sequencing library was then validated on the Agilent TapeStation (Agilent Technologies, Palo Alto, CA, USA) and quantified using Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) as well as by quantitative PCR (KAPA Biosystems, Wilmington, MA, USA). The sequencing libraries were multiplexed and clustered onto two flowcells, loaded onto an Illumina HiSeq 4000 instrument, according to the manufacturer's instructions, and sequenced using a 2x150bp Paired-End (PE) configuration. The HiSeq Control Software (HCS) was used to conduct image analysis and base calling. Raw sequence data (.bcl files) generated from Illumina HiSeq was converted into fastq files and de-multiplexed using Illumina bcl2fastq 2.20 software with one mismatch allowed for index sequence identification.",,,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX19452173,SRP423519,Illumina HiSeq 4000,SRS14105882,,,UBERON:0000113,post-juvenile adult stage,,liver,adult,,,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,Turtle_13_Liver_Tissue,SAMN29907583,,,,,,,,,,,01/05/2026,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limited cycle PCR being used for library enrichment. The sequencing library was then validated on the Agil

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [6]:
unique_sorted(library, "infoOrgan")

['liver']


In [7]:

# all
library.loc[:,'anatId'] = 'UBERON:0002107'
library.loc[:,'anatName'] = 'liver'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19452171,SRP423519,Illumina HiSeq 4000,SRS14105880,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,liver,juvenile - 1 year,perfect match,not documented,perfect match,M,,,8475,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,Turtle_4_Liver_Tissue,SAMN29907573,1,year,,,,,,,,,01/05/2026,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limited cycle PCR being used for library enrichment. The sequencing library was then validated on the Agilent TapeStation (Agilent Technologies, Palo Alto, CA, USA) and quantified using Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) as well as by quantitative PCR (KAPA Biosystems, Wilmington, MA, USA). The sequencing libraries were multiplexed and clustered onto two flowcells, loaded onto an Illumina HiSeq 4000 instrument, according to the manufacturer's instructions, and sequenced using a 2x150bp Paired-End (PE) configuration. The HiSeq Control Software (HCS) was used to conduct image analysis and base calling. Raw sequence data (.bcl files) generated from Illumina HiSeq was converted into fastq files and de-multiplexed using Illumina bcl2fastq 2.20 software with one mismatch allowed for index sequence identification.",,,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX19452173,SRP423519,Illumina HiSeq 4000,SRS14105882,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,nan,,,Turtle_13_Liver_Tissue,SAMN29907583,,,,,,,,,,,01/05/2026,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limited cycle 

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [8]:
unique_sorted(library, "infoStage")

['adult' 'hatchling' 'juvenile - 1 year']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra II Directional RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19452171,SRP423519,Illumina HiSeq 4000,SRS14105880,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,liver,juvenile - 1 year,perfect match,not documented,perfect match,M,,,8475,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_4_Liver_Tissue,SAMN29907573,1,year,,,,,,,,,01/05/2026,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limited cycle PCR being used for library enrichment. The sequencing library was then validated on the Agilent TapeStation (Agilent Technologies, Palo Alto, CA, USA) and quantified using Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) as well as by quantitative PCR (KAPA Biosystems, Wilmington, MA, USA). The sequencing libraries were multiplexed and clustered onto two flowcells, loaded onto an Illumina HiSeq 4000 instrument, according to the manufacturer's instructions, and sequenced using a 2x150bp Paired-End (PE) configuration. The HiSeq Control Software (HCS) was used to conduct image analysis and base calling. Raw sequence data (.bcl files) generated from Illumina HiSeq was converted into fastq files and de-multiplexed using Illumina bcl2fastq 2.20 software with one mismatch allowed for index sequence identification.",,,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX19452173,SRP423519,Illumina HiSeq 4000,SRS14105882,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_13_Liver_Tissue,SAMN29907583,,,,,,,,,,,01/05/2026,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limited cy

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-05-01'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX19452171,SRP423519,Illumina HiSeq 4000,SRS14105880,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,liver,juvenile - 1 year,perfect match,not documented,perfect match,M,,,8475,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_4_Liver_Tissue,SAMN29907573,1,year,,,,,,,,SAC,2026-05-01,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limited cycle PCR being used for library enrichment. The sequencing library was then validated on the Agilent TapeStation (Agilent Technologies, Palo Alto, CA, USA) and quantified using Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) as well as by quantitative PCR (KAPA Biosystems, Wilmington, MA, USA). The sequencing libraries were multiplexed and clustered onto two flowcells, loaded onto an Illumina HiSeq 4000 instrument, according to the manufacturer's instructions, and sequenced using a 2x150bp Paired-End (PE) configuration. The HiSeq Control Software (HCS) was used to conduct image analysis and base calling. Raw sequence data (.bcl files) generated from Illumina HiSeq was converted into fastq files and de-multiplexed using Illumina bcl2fastq 2.20 software with one mismatch allowed for index sequence identification.",,,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX19452173,SRP423519,Illumina HiSeq 4000,SRS14105882,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_13_Liver_Tissue,SAMN29907583,,,,,,,,,,SAC,2026-05-01,"RNA samples were shipped to GENEWIZ for analysis as follows. The Qubit 2.0 Fluorometer (ThermoFisher Scientific, Waltham, MA, USA) was used to quantify RNA concentration, and a 4200 TapeStation (Agilent Technologies, Palo Alto, CA, USA) was used to measure RNA integrity. Next, an ERCC RNA Spike-In Mix kit (ThermoFisher Scientific; 4456740) was added to normalize total RNA prior to library preparation. A strand-specific RNA sequencing library was prepared using NEBNext Ultra II Directional RNA Library Prep Kit for Illumina (NEB, Ipswich, MA, USA). Then, the enriched RNAs were fragmented at 94C for 8 minutes. First-strand and second-strand cDNA were subsequently synthesized, with the second strand of cDNA marked by incorporating dUTP during the synthesis (which quenched the amplification of the second strand, helping preserve the strand specificity). cDNA fragments were adenylated at 3' ends, and an indexed adapter was ligated to cDNA fragments, with a limi

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX19452171,SRP423519,Illumina HiSeq 4000,SRS14105880,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,liver,juvenile - 1 year,perfect match,not documented,perfect match,M,,,8475,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_4_Liver_Tissue,SAMN29907573,1,year,,,,,"PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues",,,SAC,2026-05-01
1,SRX19452173,SRP423519,Illumina HiSeq 4000,SRS14105882,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_13_Liver_Tissue,SAMN29907583,,,,,,,"PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues",,,SAC,2026-05-01
2,SRX19452175,SRP423519,Illumina HiSeq 4000,SRS14105884,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_14_Liver_Tissue,SAMN29907585,,,,,,,"PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues",,,SAC,2026-05-01
3,SRX19452177,SRP423519,Illumina HiSeq 4000,SRS14105885,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,hatchling,perfect match,not documented,perfect match,F,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_8_Liver_Tissue,SAMN29907575,,,,,,,"PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues",,,SAC,2026-05-01
4,SRX19452179,SRP423519,Illumina HiSeq 4000,SRS14105888,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,hatchling,perfect match,not documented,perfect match,F,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_9_Liver_Tissue,SAMN29907577,,,,,,,"PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues",,,SAC,2026-05-01
5,SRX19452181,SRP423519,Illumina HiSeq 4000,SRS14105890,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,hatchling,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_11_Liver_Tissue,SAMN29907579,,,,,,,"PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues",,,SAC,2026-05-01
6,SRX19452183,SRP423519,Illumina HiSeq 4000,SRS14105892,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_12_Liver_Tissue,SAMN29907581,,,,,,,"PMID: 38388772, this appears to be set up in SRA weirdly, there may be pipeline issues",,,SAC,2026-05-01


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP423519,Turtle_Hepatic_Organoids,RNA-seq raw data for the characterization of the first turtle hepatic organoids and matched tissues.,SRA,,,,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA931617,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

7

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP423519,Turtle_Hepatic_Organoids,RNA-seq raw data for the characterization of the first turtle hepatic organoids and matched tissues.,SRA,partial,Bgee 1K,7,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA931617,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '38388772'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10883927/'
experiment.loc[:,'DOI'] = '10.1038/s42003-024-05818-1'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP423519,Turtle_Hepatic_Organoids,RNA-seq raw data for the characterization of the first turtle hepatic organoids and matched tissues.,SRA,partial,Bgee 1K,7,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA931617,38388772,https://pmc.ncbi.nlm.nih.gov/articles/PMC10883927/,10.1038/s42003-024-05818-1,,


#### comments

In [18]:
experiment.loc[:,'comment'] = 'rejected organoid libraries including scRNA library, this appears to be set up in SRA weirdly, there may be pipeline issues'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP423519,Turtle_Hepatic_Organoids,RNA-seq raw data for the characterization of the first turtle hepatic organoids and matched tissues.,SRA,partial,Bgee 1K,7,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA931617,38388772,https://pmc.ncbi.nlm.nih.gov/articles/PMC10883927/,10.1038/s42003-024-05818-1,,"rejected organoid libraries including scRNA library, this appears to be set up in SRA weirdly, there may be pipeline issues"


#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62776,SRX18959304,SRP416230,Illumina NovaSeq 6000,SRS16382262,UBERON:0003665,post-anal tail muscle,UBERON:0000104,life cycle,,tail muscle,NA,perfect match,not documented,perfect match,NA,,,8479,NEBNext mRNA Library Prep,full_length,polyA,,,JG_2_WJ,SAMN32620823,,,,,,,PMID: 37502307,,,SAC,2026-04-30
62777,SRX18959303,SRP416230,Illumina NovaSeq 6000,SRS16382261,UBERON:0003665,post-anal tail muscle,UBERON:0000104,life cycle,,tail muscle,NA,perfect match,not documented,perfect match,NA,,,8479,NEBNext mRNA Library Prep,full_length,polyA,,,JG_1_WJ,SAMN32620822,,,,,,,PMID: 37502307,,,SAC,2026-04-30
62778,SRX19452171,SRP423519,Illumina HiSeq 4000,SRS14105880,UBERON:0002107,liver,UBERON:0034919,juvenile stage,,liver,juvenile - 1 year,perfect match,not documented,perfect match,M,,,8475,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_4_Liver_Tissue,SAMN29907573,1,year,,,,,"PMID: 38388772, this appears to be set up in S...",,,SAC,2026-05-01
62779,SRX19452173,SRP423519,Illumina HiSeq 4000,SRS14105882,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_13_Liver_Tissue,SAMN29907583,,,,,,,"PMID: 38388772, this appears to be set up in S...",,,SAC,2026-05-01
62780,SRX19452175,SRP423519,Illumina HiSeq 4000,SRS14105884,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,liver,adult,perfect match,not documented,perfect match,M,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_14_Liver_Tissue,SAMN29907585,,,,,,,"PMID: 38388772, this appears to be set up in S...",,,SAC,2026-05-01
62781,SRX19452177,SRP423519,Illumina HiSeq 4000,SRS14105885,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,hatchling,perfect match,not documented,perfect match,F,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_8_Liver_Tissue,SAMN29907575,,,,,,,"PMID: 38388772, this appears to be set up in S...",,,SAC,2026-05-01
62782,SRX19452179,SRP423519,Illumina HiSeq 4000,SRS14105888,UBERON:0002107,liver,UBERON:0007221,neonate stage,,liver,hatchling,perfect match,not documented,perfect match,F,,,8479,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,polyA,,,Turtle_9_Liver_Tissue,SAMN29907577,,,,,,,"PMID: 38388772, this appears to be set up in S...",,,SAC,2026-05-01


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1216,SRP277352,Chrysemys picta bellii CR and AER Transcriptome,Transcriptomic comparison of the CR and AER in...,SRA,total,Bgee 1K,12,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA656550,32862496,https://onlinelibrary.wiley.com/doi/full/10.11...,10.1111/ede.12351,,
1217,SRP416230,Transcriptome sequencing of Chrysemys picta an...,Genomic analysis provides insights into evolut...,SRA,total,Bgee 1K,6,NEBNext mRNA Library Prep,full_length,,PRJNA919086,37502307,https://pmc.ncbi.nlm.nih.gov/articles/PMC10368...,10.1002/ece3.10361,,
1218,SRP423519,Turtle_Hepatic_Organoids,RNA-seq raw data for the characterization of t...,SRA,partial,Bgee 1K,7,NEBNext Ultra II Directional RNA Library Prep Kit,full_length,,PRJNA931617,38388772,https://pmc.ncbi.nlm.nih.gov/articles/PMC10883...,10.1038/s42003-024-05818-1,,rejected organoid libraries including scRNA li...


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [30]:
! git add $git_experiment_path $git_library_path

In [31]:
! git commit -m $commit_message_exp

[develop 544dfab] adding annotated bulk experiment SRP423519
 2 files changed, 8 insertions(+)


In [32]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.24 KiB | 2.24 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   e48c050..544dfab  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push